**Calcular indicadores geoespaciales de Clima**

# Setup

In [1]:
# define root path of project 
from pathlib import Path
import sys

ROOT = Path("..").resolve()
sys.path.append(str(ROOT))

# load general setup
from utils.setup_general import *

# load GIS setup
from utils.setup_gis_python import *


Setup general cargado
Setup GIS cargado


# Parámetros

In [2]:
# Reproyectar ambas capas a un CRS apropiado para Colombia
# Se va a proyectar en MAGNA-SIRGAS que es el estándar en Colombia
# Definir 
#crs_area = "EPSG:3116"

# Definir archivo de salida
El archivo de salida se estructura para conservar la misma estructura entre todas las bases de datos

In [3]:
ORDEN_DF

['codigo_dane_municipio',
 'anno',
 'nombre_variable',
 'variable_sujeto',
 'variable_medicion',
 'variable_detalle',
 'variable_descripcion',
 'valor',
 'clasificacion_econometria']

In [4]:
# Definir DF con la estructura acordada para el proyecto
# La variable global ORDEN_DF ya tiene la lista de todas las columnas ordenadas
# Inicializar el DF vacío
dfOutput_indicadoresGeoespacialesClima = pd.DataFrame(
    columns=[ORDEN_DF]
)

# Poligonos de municipios

In [5]:
# rutas de poligonos
ruta_municipios = DATA/"raw/baa_DANE_poligonosMpiosDptos/ADMINISTRATIVO/MGN_ADM_MPIO_GRAFICO.shp"
ruta_departamentos = DATA/"raw/baa_DANE_poligonosMpiosDptos/ADMINISTRATIVO/MGN_ADM_DPTO_POLITICO.shp"

# cargar geo-data-frames
gdf_municipios = gpd.read_file(ruta_municipios)
gdf_departamentos = gpd.read_file(ruta_departamentos)


# nombre de la columna del codigo del municipio en gdf_municipios
col_id  = 'mpio_cdpmp'

# Validaciones
if gdf_municipios.crs is None:
    raise ValueError("El GeoDataFrame de municipios no tiene CRS definido.")

if gdf_municipios[col_id].isna().any() or gdf_municipios[col_id].duplicated().any():
    raise ValueError("Debe haber un código único y no vacío por municipio")

# Entender datos Raster

In [7]:
ruta_precipitacion = DATA/"raw/aca_CHIRPS_precipitacion/chirps-v3.0.1981.01.tif"


# Explorar los objetos de raster
with rasterio.open(ruta_precipitacion) as src:
    print("Número de bandas:", src.count)
    print("Filas:", src.height)
    print("Columnas:", src.width)
    print("Tipo de dato por banda:", src.dtypes)

    print("Sistema de coordenadas:", src.crs)
    print("Tamaño del píxel:", src.res)
    print("Límites:", src.bounds)
    print("Transformación:", src.transform)

    print("Valor sin información:", src.nodata)
    print("Descripción de bandas:", src.descriptions)
    
    # Extraer una banda
    banda = src.read(1, masked=True) # Las bandas de Rasterio empiezan en 1

# Explorar la banda
print(MSC_SEPARADOR, "Banda:")
print("Shape:", banda.shape)  # (filas, columnas)
print("Primer entrada:", banda[0, 0])  # Valor de la primera fila y primera columna

Número de bandas: 1
Filas: 1900
Columnas: 1720
Tipo de dato por banda: ('float32',)
Sistema de coordenadas: EPSG:4326
Tamaño del píxel: (0.05000000074505806, 0.05000000074505806)
Límites: BoundingBox(left=-120.0, bottom=-60.00000141561031, right=-33.99999871850014, top=35.0)
Transformación: | 0.05, 0.00,-120.00|
| 0.00,-0.05, 35.00|
| 0.00, 0.00, 1.00|
Valor sin información: None
Descripción de bandas: (None,)

-------------------------------- Banda:
Shape: (1900, 1720)
Primer entrada: 76.876816


# Ejemplo para un solo año

In [8]:

# Repetir para cada año

# ruta del archivo
ruta_precipitacion = DATA/"raw/aca_CHIRPS_precipitacion/chirps-v3.0.1981.01.tif"


with rasterio.open(ruta_precipitacion) as src:
    
    # Revisar que el raster tengo infor geografica
    if src.crs is None or not src.crs.is_geographic: 
        raise ValueError("Este ejemplo requiere un ráster en coordenadas geográficas")
    
    # Transformar los polígonos al CRS del ráster
    municipios_raster = gdf_municipios[[col_id, "geometry"]].to_crs(src.crs)
    
    # Media ponderada por el área cubierta de cada píxel
    resultados = exact_extract(
        src,
        municipios_raster,
        ["precipitacion_media=mean(coverage_weight=area_spherical_m2)"],
        include_cols=[col_id],
        output="pandas",
    )

# Incorporar el resultado conservando las geometrías y el CRS originales


In [9]:
resultados.head()

,mpio_cdpmp,precipitacion_media
0,05001,22.61
1,05002,41.25
2,05004,53.21
3,05021,52.37
4,05030,33.33


# CHIRPS - Precipitación

In [10]:
# Carpeta de los archivos
ruta_carpeta = DATA / "raw/aca_CHIRPS_precipitacion"

# DF con los periodos en los que tengo informacion
periodos = pd.date_range(
    start="2000-01-01",
    end="2026-08-01",
    freq="MS",
)

# Comprobar que estén todos los archivos antes de procesarlos ----------------

# Crear los nombres de los archivos
rutas = [ruta_carpeta / f"chirps-v3.0.{fecha.year}.{fecha.month:02d}.tif" for fecha in periodos]

# Revisar faltantes
faltantes = [ruta.name for ruta in rutas if not ruta.exists()]

# Reportar faltantes
if faltantes:
    raise FileNotFoundError(
        f"Faltan {len(faltantes)} archivos:\n" + "\n".join(faltantes)
    )
    
    
# Acumular resultados mensuales ---------------------------------------------

# Lista para almacenar resutlados
resultados_mensuales = []

# Reutilizar los polígonos transformados mientras el CRS no cambie
crs_anterior = None
municipios_raster = None


for fecha, ruta in tqdm(zip(periodos, rutas)):
    
    # Abrir temporalmente el archivo mensual
    with rasterio.open(ruta) as src:
        
        # Revisar que el ráster tenga información geográfica
        if src.crs is None or not src.crs.is_geographic:
            raise ValueError(f"{ruta.name}: se requiere un CRS geográfico.")
            
        # Revisar si el crs del raster es diferente al del anterior archivo
        if src.crs != crs_anterior:
            # transformar poligonos a las nuevas coordenadas
            municipios_raster = gdf_municipios[[col_id, "geometry"]].to_crs(src.crs)
            crs_anterior = src.crs # actualizar variable auxiliar para el proximo mes
        
        # Calcular media espacila ponderada por área
        resultados = exact_extract(
            src,
            municipios_raster,
            ["precipitacion_media=mean(coverage_weight=area_spherical_m2)"],
            include_cols=[col_id],
            output="pandas",
        )
    
    # Identificar el periodo del archivo
    resultados["anno"] = fecha.year
    resultados["mes"] = fecha.month
    resultados["fecha"] = fecha
    
    # Almacenar resultados mensuales
    resultados_mensuales.append(resultados)
    

Extrayendo precipitación: 0it [00:00, ?it/s]

1981-01-01 00:00:00


Extrayendo precipitación: 1it [00:21, 21.15s/it]

1981-02-01 00:00:00


Extrayendo precipitación: 2it [00:42, 21.06s/it]

1981-03-01 00:00:00


Extrayendo precipitación: 3it [01:07, 22.54s/it]


In [12]:
# Consolidar el panel municipio-mes, sin geometría
panel_precipitacion = (
    pd.concat(resultados_mensuales, ignore_index=True)
    [[col_id, "anno", "mes", "fecha", "precipitacion_media"]]
    .sort_values([col_id, "fecha"])
    .reset_index(drop=True)
)

# Explorar panel
print(f"Municipios: {panel_precipitacion[col_id].nunique():,}")
print(f"Meses: {panel_precipitacion['fecha'].nunique()}")
print(f"Filas: {len(panel_precipitacion):,}")

# mostrar el panel
display(panel_precipitacion.head())



# Cada iteracion toma entre 20-40 segundos
# Como en total son 548 meses, el tiempo de ejecución varía entre 3 - 6 horas
# Si lo ejecuto solo para el periodo de interés i.e. 2006-2014, el tiempo sería menor pero seguiría siendo importante
# por eso, voy a exportar los resultados parciales
ruta_respaldo = DATA / "intermediate/e1011_precipitacion_temporal.pkl"
panel_precipitacion.to_pickle(ruta_respaldo)

Municipios: 1,122
Meses: 3
Filas: 3,366


,mpio_cdpmp,anno,mes,fecha,precipitacion_media
0,05001,1981,1,1981-01-01,22.61
1,05001,1981,2,1981-02-01,114.58
2,05001,1981,3,1981-03-01,156.54
3,05002,1981,1,1981-01-01,41.25
4,05002,1981,2,1981-02-01,91.71


# CHIRTS - Temperatura máxima

In [17]:
# Carpeta de los archivos
ruta_carpeta = DATA / "raw/acb_CHIRTS_temperatura"

# DF con los periodos en los que tengo informacion
periodos = pd.date_range(
    start="2026-07-01",
    end="2026-08-01",
    freq="MS",
)

# Comprobar que estén todos los archivos antes de procesarlos ----------------

# Crear los nombres de los archivos
rutas = [ruta_carpeta / f"CHIRTS-ERA5.monthly_Tmax.{fecha.year}.{fecha.month:02d}.tif" for fecha in periodos]

# Revisar faltantes
faltantes = [ruta.name for ruta in rutas if not ruta.exists()]

# Reportar faltantes
if faltantes:
    raise FileNotFoundError(
        f"Faltan {len(faltantes)} archivos:\n" + "\n".join(faltantes)
    )
    
    
# Acumular resultados mensuales ---------------------------------------------

# Lista para almacenar resutlados
resultados_mensuales = []

# Reutilizar los polígonos transformados mientras el CRS no cambie
crs_anterior = None
municipios_raster = None


for fecha, ruta in tqdm(zip(periodos, rutas)):
    
    # Abrir temporalmente el archivo mensual
    with rasterio.open(ruta) as src:
        
        # Revisar que el ráster tenga información geográfica
        if src.crs is None or not src.crs.is_geographic:
            raise ValueError(f"{ruta.name}: se requiere un CRS geográfico.")
            
        # Revisar si el crs del raster es diferente al del anterior archivo
        if src.crs != crs_anterior:
            # transformar poligonos a las nuevas coordenadas
            municipios_raster = gdf_municipios[[col_id, "geometry"]].to_crs(src.crs)
            crs_anterior = src.crs # actualizar variable auxiliar para el proximo mes
        
        # Calcular media espacila ponderada por área
        resultados = exact_extract(
            src,
            municipios_raster,
            ["Tmax_media=mean(coverage_weight=area_spherical_m2)"],
            include_cols=[col_id],
            output="pandas",
        )
    
    # Identificar el periodo del archivo
    resultados["anno"] = fecha.year
    resultados["mes"] = fecha.month
    resultados["fecha"] = fecha
    
    # Almacenar resultados mensuales
    resultados_mensuales.append(resultados)
    

2it [00:44, 22.06s/it]


In [16]:
# Consolidar el panel municipio-mes, sin geometría
panel_Tmax = (
    pd.concat(resultados_mensuales, ignore_index=True)
    [[col_id, "anno", "mes", "fecha", "Tmax_media"]]
    .sort_values([col_id, "fecha"])
    .reset_index(drop=True)
)

# Explorar panel
print(f"Municipios: {panel_Tmax[col_id].nunique():,}")
print(f"Meses: {panel_Tmax['fecha'].nunique()}")
print(f"Filas: {len(panel_Tmax):,}")

# mostrar el panel
display(panel_Tmax.head())



# Cada iteracion toma entre 20-40 segundos
# Como en total son 548 meses, el tiempo de ejecución varía entre 3 - 6 horas
# Si lo ejecuto solo para el periodo de interés i.e. 2006-2014, el tiempo sería menor pero seguiría siendo importante
# por eso, voy a exportar los resultados parciales
ruta_respaldo = DATA / "intermediate/e1011_Tmax_temporal.pkl"
panel_Tmax.to_pickle(ruta_respaldo)

Municipios: 1,122
Meses: 3
Filas: 3,366


,mpio_cdpmp,anno,mes,fecha,Tmax_media
0,05001,2026,7,2026-07-01,23.41
1,05001,2026,8,2026-08-01,23.36
2,05002,2026,7,2026-07-01,23.98
3,05002,2026,8,2026-08-01,23.86
4,05004,2026,7,2026-07-01,22.38


# CHIRTS - Temperatura mínima

In [ ]:
# Carpeta de los archivos
ruta_carpeta = DATA / "raw/acb_CHIRTS_temperatura"

# DF con los periodos en los que tengo informacion
periodos = pd.date_range(
    start="2000-01-01",
    end="2026-08-01",
    freq="MS",
)

# Comprobar que estén todos los archivos antes de procesarlos ----------------

# Crear los nombres de los archivos
rutas = [ruta_carpeta / f"CHIRTS-ERA5.monthly_Tmin.{fecha.year}.{fecha.month:02d}.tif" for fecha in periodos]

# Revisar faltantes
faltantes = [ruta.name for ruta in rutas if not ruta.exists()]

# Reportar faltantes
if faltantes:
    raise FileNotFoundError(
        f"Faltan {len(faltantes)} archivos:\n" + "\n".join(faltantes)
    )
    
    
# Acumular resultados mensuales ---------------------------------------------

# Lista para almacenar resutlados
resultados_mensuales = []

# Reutilizar los polígonos transformados mientras el CRS no cambie
crs_anterior = None
municipios_raster = None


for fecha, ruta in tqdm(zip(periodos, rutas)):
    
    # Abrir temporalmente el archivo mensual
    with rasterio.open(ruta) as src:
        
        # Revisar que el ráster tenga información geográfica
        if src.crs is None or not src.crs.is_geographic:
            raise ValueError(f"{ruta.name}: se requiere un CRS geográfico.")
            
        # Revisar si el crs del raster es diferente al del anterior archivo
        if src.crs != crs_anterior:
            # transformar poligonos a las nuevas coordenadas
            municipios_raster = gdf_municipios[[col_id, "geometry"]].to_crs(src.crs)
            crs_anterior = src.crs # actualizar variable auxiliar para el proximo mes
        
        # Calcular media espacila ponderada por área
        resultados = exact_extract(
            src,
            municipios_raster,
            ["Tmin_media=mean(coverage_weight=area_spherical_m2)"],
            include_cols=[col_id],
            output="pandas",
        )
    
    # Identificar el periodo del archivo
    resultados["anno"] = fecha.year
    resultados["mes"] = fecha.month
    resultados["fecha"] = fecha
    
    # Almacenar resultados mensuales
    resultados_mensuales.append(resultados)
    

1it [00:17, 17.03s/it]

In [16]:
# Consolidar el panel municipio-mes, sin geometría
panel_Tmin = (
    pd.concat(resultados_mensuales, ignore_index=True)
    [[col_id, "anno", "mes", "fecha", "Tmin_media"]]
    .sort_values([col_id, "fecha"])
    .reset_index(drop=True)
)

# Explorar panel
print(f"Municipios: {panel_Tmin[col_id].nunique():,}")
print(f"Meses: {panel_Tmax['fecha'].nunique()}")
print(f"Filas: {len(panel_Tmin):,}")

# mostrar el panel
display(panel_Tmax.head())



# Cada iteracion toma entre 20-40 segundos
# Como en total son 548 meses, el tiempo de ejecución varía entre 3 - 6 horas
# Si lo ejecuto solo para el periodo de interés i.e. 2006-2014, el tiempo sería menor pero seguiría siendo importante
# por eso, voy a exportar los resultados parciales
ruta_respaldo = DATA / "intermediate/e1011_Tmin_temporal.pkl"
panel_Tmin.to_pickle(ruta_respaldo)

Municipios: 1,122
Meses: 3
Filas: 3,366


,mpio_cdpmp,anno,mes,fecha,Tmax_media
0,05001,2026,7,2026-07-01,23.41
1,05001,2026,8,2026-08-01,23.36
2,05002,2026,7,2026-07-01,23.98
3,05002,2026,8,2026-08-01,23.86
4,05004,2026,7,2026-07-01,22.38
